# Chapter 7 — Learning to Rank, Deep Ranking, and Graph-Based Methods

The chapter's two deep dives on one pair of rails: a **LambdaMART** ranker with LightGBM
(section 7.3) and a **DCN-v2** ranker in PyTorch (section 7.4), trained on the same
MovieLens 25M features and evaluated on the same test groups so the head-to-head
comparison in 7.4.3 is apples-to-apples.

Structure:
1. Data and temporal split
2. Feature engineering (7.3.1 / listing 7.1)
3. Ranking groups and relevance grades (7.3.2 / listing 7.2)
4. LambdaMART training and evaluation (listings 7.3–7.4)
5. SHAP interpretation (7.3.3 / listing 7.5)
6. DCN-v2: model, training, evaluation (listings 7.6–7.9)
7. Head-to-head + cross-feature ablation (7.4.3 / listing 7.10)

Requires the book's `recsys` package on the path and the MovieLens 25M files
(`ratings.csv`, `movies.csv`).

In [ ]:
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

sys.path.insert(0, "..")  # repo root, so `recsys` imports

from recsys.fourstage_recsys.ranking import (
    FEATURE_COLS, GENRES,
    build_genre_matrix, build_user_features, build_item_features,
    temporal_split, make_ranking_dataset,
    train_lambdamart,
    per_group_metrics, compare_models, popularity_scores,
    MovieIndex, FeatureEmbedder, DCNv2Ranker, GroupedRankingDataset,
    fit_dcn, predict_scores, listwise_loss, bpr_loss,
)

try:
    import mlflow
    mlflow.set_experiment("ch07-ranking")
    MLFLOW = True
except ImportError:
    MLFLOW = False

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| mlflow:", MLFLOW)

## 1. Data and temporal split

We use the full 25M ratings for the book's reported numbers. For a laptop-friendly run,
set `SAMPLE_USERS` to keep a random subset of users — the pipeline is identical, only
the numbers shrink. The split is *global-temporal* (chapter 4): the most recent 10% of
all ratings are the test period, the 10% before that validation. No model ever sees the
future.

In [ ]:
DATA_DIR = Path("../data/ml-25m")   # adjust to your checkout
SAMPLE_USERS = 20_000                # None = full 25M run

ratings = pd.read_csv(DATA_DIR / "ratings.csv")
movies = pd.read_csv(DATA_DIR / "movies.csv")

if SAMPLE_USERS is not None:
    keep = rng.choice(ratings["userId"].unique(), size=SAMPLE_USERS, replace=False)
    ratings = ratings[ratings["userId"].isin(keep)]

train, valid, test = temporal_split(ratings, valid_fraction=0.1, test_fraction=0.1)
print(f"train {len(train):,}  valid {len(valid):,}  test {len(test):,}")
print("train ends:", pd.to_datetime(train.timestamp.max(), unit='s').date(),
      "| test starts:", pd.to_datetime(test.timestamp.min(), unit='s').date())

## 2. Feature engineering (listing 7.1)

Three families — user, item, cross — computed **from the training window only**. The
builders take `train` explicitly; if you find yourself passing anything else, that's the
temporal-leakage alarm going off.

In [ ]:
genre_matrix = build_genre_matrix(movies)
user_features = build_user_features(train, genre_matrix)
item_features = build_item_features(train, genre_matrix)

print(len(FEATURE_COLS), "features")
user_features.head(3)

## 3. Ranking groups and relevance grades (listing 7.2)

Each user becomes a *group*: their rated movies in the window (graded 0–3) plus 100
sampled never-seen negatives (grade 0). The negatives are the point — at serving time
the ranker scores retrieval candidates the user has never touched, so it must train on
that distribution too.

In [ ]:
N_NEGATIVES = 100

frames, X, y, groups = {}, {}, {}, {}
for name, window in [("train", train), ("valid", valid), ("test", test)]:
    frames[name], X[name], y[name], groups[name] = make_ranking_dataset(
        window, train, user_features, item_features,
        n_negatives=N_NEGATIVES, seed=SEED,
    )
    print(f"{name:5s}: {len(X[name]):>9,} rows  {len(groups[name]):>7,} groups  "
          f"pos rate {(y[name] > 0).mean():.3f}")

## 4. LambdaMART with LightGBM (listing 7.3)

The `lambdarank` objective computes, per boosting round and per group, pairwise
gradients scaled by |ΔNDCG| of the swap — swaps near the top of the list get large
gradients. Early stopping on validation NDCG@10, the same stopping rule the deep model
will use.

In [ ]:
t0 = time.time()
ranker = train_lambdamart(
    X["train"], y["train"], groups["train"],
    X["valid"], y["valid"], groups["valid"],
)
lgb_seconds = time.time() - t0
print(f"trained in {lgb_seconds:.0f}s (CPU), best iteration {ranker.best_iteration}")

### Evaluation against the baselines (listing 7.4)

Popularity is chapter 2's baseline dressed as a ranker. If you have the chapter 5
two-tower checkpoint, add its dot-product scores as a third row — that comparison
isolates exactly what the ranking stage buys over retrieval scores.

In [ ]:
lgb_scores = ranker.predict(X["test"])

report = compare_models(
    {
        "Popularity": popularity_scores(frames["test"]),
        "LambdaMART": lgb_scores,
        # "Two-tower (ch5)": twotower_scores,   # plug in your ch5 checkpoint here
    },
    y["test"], groups["test"],
)
report.round(4)

## 5. Interpretation with SHAP (listing 7.5)

Global view first (which features matter, in which direction), then one individual
prediction. Expect the engineered crosses — `x_genre_affinity` above all — at the top:
the empirical backing for section 7.3.1's claim that feature engineering is where the
value lives, and the setup for the DCN question (*could a model learn these crosses
itself?*).

In [ ]:
import shap

explainer = shap.TreeExplainer(ranker)
sample = X["test"].iloc[:2000]
shap_values = explainer.shap_values(sample)

shap.summary_plot(shap_values, sample, feature_names=FEATURE_COLS, max_display=15)

In [ ]:
row = X["test"].iloc[[1337]]
shap.force_plot(explainer.expected_value, explainer.shap_values(row), row,
                matplotlib=True)

## 6. DCN-v2 in PyTorch (listings 7.6–7.9)

Same rows, same features — plus the one thing trees can't digest: the raw `movieId` as
a 32-dim embedding. Cross network and deep network in parallel, sampled-softmax
listwise loss, early stopping on validation NDCG@10. Swap `listwise_loss` for
`bpr_loss` in `fit_dcn` to feel section 7.2.1 as a design choice — it's one argument.

In [ ]:
movie_index = MovieIndex(train["movieId"].unique())

embedder = FeatureEmbedder(
    {"movie": movie_index.cardinality}, n_dense=len(FEATURE_COLS), emb_dim=32,
)
model = DCNv2Ranker(embedder, n_cross_layers=3, rank=64, deep_dims=(256, 128, 64))
print(sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")

train_ds = GroupedRankingDataset(
    frames["train"], groups["train"], movie_index, n_negatives=8, seed=SEED,
)
print(f"{len(train_ds):,} training examples (1 positive + 8 negatives each)")

In [ ]:
t0 = time.time()
result = fit_dcn(
    model, train_ds,
    frames["valid"], y["valid"], groups["valid"],
    movie_index, device,
    epochs=20, patience=3, batch_size=256,
    loss_fn=listwise_loss,          # <- try bpr_loss here
)
dcn_seconds = time.time() - t0
print(f"trained in {dcn_seconds:.0f}s on {device}, best epoch {result['best_epoch']}")

## 7. Head-to-head (listing 7.10)

Two score vectors, one shared test set — nothing else differs. One small table that
settles a lot of arguments.

In [ ]:
dcn_scores = predict_scores(model, frames["test"], movie_index, device)

report = compare_models(
    {
        "Popularity": popularity_scores(frames["test"]),
        "LambdaMART": lgb_scores,
        "DCN-v2": dcn_scores,
    },
    y["test"], groups["test"],
)
report["train_seconds"] = [np.nan, lgb_seconds, dcn_seconds]

if MLFLOW:
    with mlflow.start_run(run_name="ch07-head-to-head"):
        mlflow.log_params({"n_negatives": N_NEGATIVES, "sample_users": SAMPLE_USERS})
        for name, row in report.iterrows():
            mlflow.log_metric(f"{name}_ndcg10", row["ndcg@10"])
            mlflow.log_metric(f"{name}_mrr", row["mrr"])

report.round(4)

### How much did the ranker actually change the list?

The 7.1 promise: count it. If the ranker's top-10 is retrieval's top-10 in a trench coat
(`overlap` near 1, `mean_shift` near 0), you're paying model costs for an `argsort` you
already had. Substitute the chapter 5 retrieval scores for `popularity_scores` below to
run the real comparison.

In [ ]:
from recsys.fourstage_recsys.ranking import list_divergence

baseline = popularity_scores(frames["test"])   # <- replace with ch5 retrieval scores
for name, scores in [("LambdaMART", lgb_scores), ("DCN-v2", dcn_scores)]:
    print(name, list_divergence(baseline, scores, groups["test"], k=10))

### Ablation: who needs the hand-crafted crosses?

The interesting third row of section 7.4.3: retrain both models with the three
engineered cross features **removed** from the inputs. The prediction from the chapter —
LambdaMART degrades noticeably (the trees were told the multiplications), DCN-v2 barely
flinches (the cross network learns them) — is now a cell away from being your finding
instead of my claim.

In [ ]:
from recsys.fourstage_recsys.ranking.features import CROSS_COLS

NO_CROSS = [f for f in FEATURE_COLS if f not in CROSS_COLS]

# Trees without crosses
ranker_nc = train_lambdamart(
    X["train"][NO_CROSS], y["train"], groups["train"],
    X["valid"][NO_CROSS], y["valid"], groups["valid"],
)
lgb_nc_scores = ranker_nc.predict(X["test"][NO_CROSS])

# DCN without crosses
embedder_nc = FeatureEmbedder(
    {"movie": movie_index.cardinality}, n_dense=len(NO_CROSS), emb_dim=32,
)
model_nc = DCNv2Ranker(embedder_nc, n_cross_layers=3, rank=64, deep_dims=(256, 128, 64))
train_ds_nc = GroupedRankingDataset(
    frames["train"], groups["train"], movie_index,
    n_negatives=8, feature_cols=NO_CROSS, seed=SEED,
)
fit_dcn(model_nc, train_ds_nc, frames["valid"], y["valid"], groups["valid"],
        movie_index, device, feature_cols=NO_CROSS,
        epochs=20, patience=3, verbose=False)
dcn_nc_scores = predict_scores(model_nc, frames["test"], movie_index, device,
                               feature_cols=NO_CROSS)

compare_models(
    {
        "LambdaMART": lgb_scores,
        "LambdaMART (no crosses)": lgb_nc_scores,
        "DCN-v2": dcn_scores,
        "DCN-v2 (no crosses)": dcn_nc_scores,
    },
    y["test"], groups["test"],
).round(4)

---
The final numbers from the full 25M run go into the chapter's two `[TBD]` tables
(sections 7.3.3 and 7.4.3), along with the training times printed above.